In [2]:
# %%
# # 동반이환 만성질환 계약자의 개인화 건강 개입 경로 생성 모형
# 분석 흐름: XGBoost (1단계) → GPT-4o-mini 가드레일 에이전트 (2단계) → DiCE (3단계)
# 데이터: 국민건강영양조사(KNHANES) 2020–2024
# 층화 기준: 연령군(중장년 40–59세 / 고령 60세 이상) × 성별(남/여) → 4개 모델
# [해외 논문 대비 변경사항]
#   - 성별 2개 모델 → 연령×성별 교차 4개 모델
#   - age 변수 추가 (KNHANES 원시자료 변수명 확인 필요)
#   - AGEGROUP_CONFIG 딕셔너리로 4개 그룹 일괄 관리
#   - 케이스 스터디: 남녀 각 1건 → 4개 그룹 각 1건

# %%
# ## 0. 라이브러리 불러오기 및 기본 설정

import json
import joblib
import numpy as np
import optuna
import pandas as pd
import pyreadstat
import xgboost as xgb
import openai
import dice_ml

from sklearn.metrics import classification_report, f1_score
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_sample_weight

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 100)
pd.set_option('display.max_colwidth', None)
optuna.logging.set_verbosity(optuna.logging.WARNING)

# OpenAI API 키 설정
OPENAI_API_KEY = ""   # ← API 키 입력
client = openai.OpenAI(api_key=OPENAI_API_KEY)

# 4개 층화 그룹 정의 (연령 하한·상한 inclusive, 성별코드: 1=남 2=여)
AGEGROUP_CONFIG = {
    "중장년_남성": {"age_min": 40, "age_max": 59, "sex_code": 1.0},
    "중장년_여성": {"age_min": 40, "age_max": 59, "sex_code": 2.0},
    "고령_남성":   {"age_min": 60, "age_max": 99, "sex_code": 1.0},
    "고령_여성":   {"age_min": 60, "age_max": 99, "sex_code": 2.0},
}

# %%
# ## 1. 국민건강영양조사(KNHANES) 2020–2024 데이터 병합

df20, _ = pyreadstat.read_sas7bdat('hn20_all.sas7bdat')
df21, _ = pyreadstat.read_sas7bdat('hn21_all.sas7bdat')
df22, _ = pyreadstat.read_sas7bdat('hn22_all.sas7bdat')
df23, _ = pyreadstat.read_sas7bdat('hn23_all.sas7bdat')
df24, _ = pyreadstat.read_sas7bdat('hn24_all.sas7bdat')

# %%
# ### 1-1. 분석 변수 선택
# 해외 논문 대비 'age' 변수 추가 — 연령군 층화에 필수

KEY_COLS    = ['ID', 'year', 'sex', 'age']   # age 추가
CAT_COLS    = [
    'HE_obe', 'BO1_1', 'BO1_2', 'BO1_3', 'BD1_11', 'BD2_1', 'BS3_1',
    'BE3_71', 'BE3_75', 'BE3_81', 'BE3_91', 'pa_aerobic', 'L_BR_FQ',
    'BP1', 'mh_stress', 'incm', 'ho_incm', 'edu', 'BH1',
]
NUM_COLS    = [
    'HE_BMI', 'HE_wc', 'HE_wt',
    'N_EN', 'N_CHO', 'N_SUGAR', 'N_NA', 'N_FAT',
    'N_SFA', 'N_TDF', 'N_K', 'N_PROT',
]
TARGET_COLS = ['HE_DM_HbA1c', 'HE_HP']
ALL_VARS    = KEY_COLS + CAT_COLS + NUM_COLS + TARGET_COLS

# %%
# ### 1-2. 연도별 데이터 필터링 후 병합
# 연도별로 존재하는 컬럼만 선택 (변수 누락 방지)

df_total = pd.concat(
    [d[[v for v in ALL_VARS if v in d.columns]].copy()
     for d in [df20, df21, df22, df23, df24]],
    axis=0
).reset_index(drop=True)

# %%
# ### 1-3. 목표 변수 이진 인코딩
# HE_DM_HbA1c: 1=정상→0, 3=당뇨→1
# HE_HP:        1=정상→0, 4=고혈압→1

df_total['HE_DM_HbA1c'] = df_total['HE_DM_HbA1c'].map({1: 0, 3: 1}).fillna(-999)
df_total['HE_HP']        = df_total['HE_HP'].map({1: 0, 4: 1}).fillna(-999)

for col in ['HE_DM_HbA1c', 'HE_HP']:
    print(f"\n{col} 분포:\n", df_total[col].value_counts())

# %%
# ### 1-4. 4개 위험등급 통합 레이블 생성
# 등급 0: 정상 | 등급 1: 고혈압만 | 등급 2: 당뇨만 | 등급 3: 동반이환(고위험)

conditions = [
    (df_total['HE_DM_HbA1c'] == 0) & (df_total['HE_HP'] == 0),
    (df_total['HE_DM_HbA1c'] == 0) & (df_total['HE_HP'] == 1),
    (df_total['HE_DM_HbA1c'] == 1) & (df_total['HE_HP'] == 0),
    (df_total['HE_DM_HbA1c'] == 1) & (df_total['HE_HP'] == 1),
]
df_total['integrated_target'] = np.select(conditions, [0, 1, 2, 3], default=np.nan)
df_total = df_total.dropna().reset_index(drop=True)
df_total['integrated_target'] = df_total['integrated_target'].astype(int)

print("\n위험등급 분포 (전체):")
print(df_total['integrated_target'].value_counts().sort_index())

# %%
# ### 1-5. 연령군 변수 생성
# 중장년(40–59세) = 0 / 고령(60세 이상) = 1
# 40세 미만은 분석 대상에서 제외 (보험 언더라이팅 실무 기준)

if 'age' not in df_total.columns:
    raise ValueError("'age' 변수가 없습니다. KNHANES 원시자료 변수명(예: age, HE_age)을 확인하세요.")

df_total['age'] = pd.to_numeric(df_total['age'], errors='coerce')
df_total = df_total[df_total['age'] >= 40].copy().reset_index(drop=True)
df_total['age_group'] = np.where(df_total['age'] < 60, 0, 1)  # 0=중장년, 1=고령

print("\n연령군 분포 (0=중장년 40–59세, 1=고령 60세+):")
print(df_total['age_group'].value_counts().sort_index())

# %%
# ### 1-6. 컬럼명 변환: SAS 원시 코드 → 영문 레이블 (원본과 동일)

ENGLISH_LABEL_DICT = {
    'ID':           'ID',
    'year':         'SurveyYear',
    'sex':          'Sex',
    'age':          'Age',
    'age_group':    'AgeGroup',       # 0=중장년, 1=고령 (신규)
    'HE_DM_HbA1c':  'Diabetes',
    'HE_HP':        'Hypertension',
    'integrated_target': 'RiskClass',
    'HE_obe':   'ObesityStatus',
    'BO1_1':    'WeightChangeStatus',
    'BO1_2':    'WeightLossAmount',
    'BO1_3':    'WeightGainAmount',
    'BD1_11':   'DrinkingFrequency',
    'BD2_1':    'DrinkingAmount',
    'BS3_1':    'SmokingStatus',
    'BE3_71':   'VigorousActivity_Work',
    'BE3_75':   'VigorousActivity_Leisure',
    'BE3_81':   'ModerateActivity_Work',
    'BE3_91':   'WalkingActivity',
    'pa_aerobic': 'AerobicActivityRate',
    'L_BR_FQ':  'BreakfastFrequency',
    'BP1':      'StressLevel',
    'mh_stress':'StressAwarenessRate',
    'incm':     'PersonalIncomeQuartile',
    'ho_incm':  'HouseholdIncomeQuartile',
    'edu':      'EducationLevel',
    'BH1':      'HealthScreeningStatus',
    'HE_BMI':   'BMI',
    'HE_wc':    'WaistCirc',
    'HE_wt':    'Weight',
    'N_EN':     'Energy_kcal',
    'N_CHO':    'Carb_g',
    'N_SUGAR':  'Sugar_g',
    'N_NA':     'Sodium_mg',
    'N_FAT':    'Fat_g',
    'N_SFA':    'SaturatedFat_g',
    'N_TDF':    'Fiber_g',
    'N_K':      'Potassium_mg',
    'N_PROT':   'Protein_g',
}

df_total.rename(columns=ENGLISH_LABEL_DICT, inplace=True)
df_total.to_csv('./knhanes_comorbid_2020_2024_age_stratified.csv',
                index=False, encoding='utf-8-sig')
print(">>> 전처리 완료. CSV 저장됨.")
df_total.head()

# %%
# ## 2. 피처 정의 및 수치형 데이터셋 구성

NUM_FEATURES = [
    'BMI', 'WaistCirc', 'Weight',
    'Energy_kcal', 'Carb_g', 'Sugar_g', 'Sodium_mg',
    'Fat_g', 'SaturatedFat_g', 'Fiber_g', 'Potassium_mg', 'Protein_g',
]

CAT_FEATURES = [
    'ObesityStatus', 'WeightChangeStatus', 'WeightLossAmount', 'WeightGainAmount',
    'DrinkingFrequency', 'DrinkingAmount', 'SmokingStatus',
    'VigorousActivity_Work', 'VigorousActivity_Leisure',
    'ModerateActivity_Work', 'WalkingActivity', 'AerobicActivityRate',
    'BreakfastFrequency', 'StressLevel', 'StressAwarenessRate',
    'PersonalIncomeQuartile', 'HouseholdIncomeQuartile',
    'EducationLevel', 'HealthScreeningStatus',
]

X_FEATURES = NUM_FEATURES + CAT_FEATURES
TARGET_COL  = 'RiskClass'

# DiCE 탐색 시 고정할 변수 (비가변: 사회경제적 요인 등)
FIXED_FEATURES = [
    'StressLevel', 'StressAwarenessRate',
    'PersonalIncomeQuartile', 'HouseholdIncomeQuartile',
    'EducationLevel', 'HealthScreeningStatus',
]
VARY_FEATURES = [f for f in X_FEATURES if f not in FIXED_FEATURES]

# 수치형 데이터셋 구성 (XGBoost·DiCE 입력용)
# AgeGroup·Sex는 층화 필터링에만 사용하고 피처에서는 제외
required_cols = X_FEATURES + [TARGET_COL, 'Sex', 'AgeGroup']
df_final = df_total[required_cols].copy()
for col in df_final.columns:
    df_final[col] = pd.to_numeric(df_final[col], errors='coerce').fillna(0)
df_final = df_final.astype(float)

print(f"\ndf_final 형태: {df_final.shape}")
print(f"문자형 컬럼 존재 여부: {(df_final.dtypes == 'object').any()}")

# 연령군 × 성별 × 위험등급 분포 확인 (논문 표 1 기초 자료)
print("\n[연령군 × 성별 × 위험등급 분포]")
dist_table = (
    df_final.groupby(['AgeGroup', 'Sex', 'RiskClass'])
            .size()
            .unstack(fill_value=0)
)
print(dist_table)
# AgeGroup: 0=중장년(40–59세), 1=고령(60세+) / Sex: 1=남, 2=여

# %%
# ## 3. 1단계 — XGBoost 연령×성별 교차 층화 모델 학습 (Optuna)
# 해외 논문: 성별 2개 모델 → 국문 논문: 연령×성별 4개 모델

def train_model(age_group: float, gender_code: float, group_name: str):
    """
    age_group: 0=중장년(40–59세), 1=고령(60세+)
    gender_code: 1.0=남성, 2.0=여성
    group_name: 로그 출력용 그룹명 (예: '중장년_남성')
    """
    print(f"\n{'='*20} [{group_name}] 모델 학습 {'='*20}")

    # 해당 연령군·성별 서브셋 추출
    df_g = df_final[
        (df_final['AgeGroup'] == age_group) &
        (df_final['Sex']      == gender_code)
    ].copy()

    print(f"  표본 수: {len(df_g)}명")
    print(f"  위험등급 분포:\n{df_g[TARGET_COL].value_counts().sort_index()}")

    # 표본 수 부족 시 경고 후 None 반환
    if len(df_g) < 100:
        print(f"  ⚠️  표본 수 부족 ({len(df_g)}명). 모델 학습을 건너뜁니다.")
        return None, None

    X = df_g[X_FEATURES]
    y = df_g[TARGET_COL].astype(int)

    # 훈련/검증 분리 (8:2, 층화 분할)
    X_train, X_val, y_train, y_val = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )
    # 클래스 불균형 보정 (동반이환 등급 3의 소수 클래스 문제)
    sw = compute_sample_weight('balanced', y=y_train)

    # Optuna 하이퍼파라미터 최적화 (가중 F1 기준)
    def objective(trial):
        params = {
            'n_estimators':     trial.suggest_int('n_estimators', 100, 500),
            'max_depth':        trial.suggest_int('max_depth', 3, 7),
            'learning_rate':    trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
            'subsample':        trial.suggest_float('subsample', 0.7, 1.0),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.7, 1.0),
            'objective':        'multi:softprob',
            'num_class':        4,
            'tree_method':      'hist',
            'random_state':     42,
        }
        mdl = xgb.XGBClassifier(**params)
        mdl.fit(X_train, y_train, sample_weight=sw)
        return f1_score(y_val, mdl.predict(X_val), average='weighted')

    study = optuna.create_study(direction='maximize')
    study.optimize(objective, n_trials=10)

    # 최적 파라미터로 최종 모델 학습
    best_model = xgb.XGBClassifier(
        **study.best_params,
        objective='multi:softprob',
        num_class=4,
        tree_method='hist',
        random_state=42,
    )
    best_model.fit(X_train, y_train, sample_weight=sw)

    print(f"\n[{group_name}] 분류 성능 보고서:")
    print(classification_report(y_val, best_model.predict(X_val)))

    return best_model, study.best_params


# 4개 그룹 모델 일괄 학습
models    = {}   # 그룹명 → 학습된 모델
params_dict = {}

for group_name, cfg in AGEGROUP_CONFIG.items():
    mdl, prm = train_model(
        age_group   = 0.0 if cfg['age_min'] < 60 else 1.0,
        gender_code = cfg['sex_code'],
        group_name  = group_name,
    )
    models[group_name]      = mdl
    params_dict[group_name] = prm

# 모델 및 설정 저장
agent_config = {
    'X_features':    X_FEATURES,
    'num_features':  NUM_FEATURES,
    'cat_features':  CAT_FEATURES,
    'target_col':    TARGET_COL,
    'agegroup_config': AGEGROUP_CONFIG,
    'best_params':   params_dict,
}
for group_name, mdl in models.items():
    if mdl is not None:
        joblib.dump(mdl, f'model_{group_name}.pkl')

joblib.dump(agent_config, 'agent_config.pkl')
joblib.dump(df_final,     'df_final.pkl')
print(">>> 모델 및 설정 저장 완료.")

# %%
# ## 4. 2단계 — GPT-4o-mini 임상 가드레일 에이전트
# 원본과 동일한 로직, 한국어 주석 추가

def get_clinical_guardrails(patient_data: pd.DataFrame,
                             group_name: str,
                             allowed_features: list,
                             metadata: dict) -> dict:
    """
    GPT-4o-mini 호출 → 환자 개인 프로파일 기반 허용 범위(permitted_range) 도출.
    반환값: {'reasoning': str, 'guardrail_ranges': {변수명: [최솟값, 최댓값]}}

    핵심 제약:
    - 나트륨 감소 시 탄수화물·당류 상한을 현재값으로 고정 (개입 충돌 방지)
    - BMI·허리둘레·체중은 반드시 같은 방향으로 변화 (생리적 일관성)
    - 에너지 섭취량 하한 = 현재값의 70% (근감소·영양결핍 방지)
    """
    patient_info = patient_data.to_dict(orient='records')[0]

    prompt = f"""
You are a high-precision Clinical Data Strategist for a health insurance company.
You must respond ONLY in valid JSON using the exact variable names listed below.

[Patient Information ({group_name})]
{json.dumps(patient_info, ensure_ascii=False, indent=1)}

[Available Variable Names]
{", ".join(allowed_features)}

[Categorical Variable Valid Values (use only these values)]
{json.dumps(metadata, ensure_ascii=False, indent=1)}

[Guardrail Principles — apply ALL of the following strictly]

1. CONFLICT PREVENTION:
   - Cap Carb_g at [0, current Carb_g] and Sugar_g at [0, current Sugar_g]
     whenever Sodium_mg is reduced below its current value.
   - This prevents compensatory carbohydrate intake that worsens glycaemic control.

2. PHYSICAL CONSISTENCY (critical — violations found in prior run):
   - BMI range: [current_BMI * 0.85, current_BMI].
     The upper bound must NOT exceed the current value; reduction is the goal.
   - WaistCirc range: [current_WaistCirc * 0.85, current_WaistCirc].
     WaistCirc MUST decrease together with BMI/Weight; it cannot decrease
     independently while BMI and Weight remain unchanged.
   - Weight range: [current_Weight * 0.85, current_Weight].
   - All three (BMI, WaistCirc, Weight) must move in the same direction.
     If only dietary composition changes (no weight loss pathway), set all three
     to [current * 0.97, current] to keep them nearly fixed but consistent.

3. ENERGY FLOOR:
   - Energy_kcal lower bound = max(current_Energy_kcal * 0.70, 500).
   - Upper bound = current_Energy_kcal (do not allow caloric increase).

4. SAFETY: All lower bounds must be >= 0.

5. DIVERSITY HINT:
   - Set Fat_g range to allow both reduction and mild increase
     so that DiCE can generate structurally distinct pathways
     (weight-loss-led vs. dietary-composition-led).

[Response Format — valid JSON only, no extra text]
{{
    "reasoning": "Detailed clinical rationale explaining each constraint choice",
    "guardrail_ranges": {{ "ExactVariableName": [min, max] }}
}}
"""

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system",
             "content": "You are an expert who responds strictly in valid JSON, "
                        "fully adhering to clinical guidelines and the provided "
                        "data schema. Never allow WaistCirc to decrease while "
                        "BMI and Weight remain unchanged."},
            {"role": "user", "content": prompt},
        ],
        response_format={"type": "json_object"},
    )
    return json.loads(response.choices[0].message.content)

# %%
# ## 5. 3단계 — 가드레일 제약 주입 DiCE 반사실 생성 + 단계적 폴백
# 원본과 동일한 로직, 한국어 주석 추가

def run_dice_pipeline(model, df_input: pd.DataFrame,
                      age_group: float, gender_code: float, group_name: str):
    """
    2단계+3단계 통합 파이프라인:
      1. 등급 3(동반이환) 환자 표본 추출
      2. LLM 가드레일 생성
      3. 물리적 보정 레이어 적용 (코드 레벨 하드 규칙)
      4. DiCE 탐색 + 단계적 폴백 (등급 0 → 1 → 2)

    age_group: 0=중장년, 1=고령
    gender_code: 1.0=남, 2.0=여
    group_name: 로그·저장용 그룹명
    """
    print(f"\n{'='*25} [{group_name}] 파이프라인 실행 {'='*25}")

    df_stable = df_input.copy().astype(float)

    # ── 1. 등급 3 환자 표본 추출 ─────────────────────────────────────────────
    # 모델 예측도 등급 3이고 BMI≥23인 확인된 동반이환 환자만 대상
    class3_pool = df_stable[
        (df_stable['AgeGroup'] == age_group) &   # 연령군 필터 (신규)
        (df_stable['Sex']      == gender_code) &
        (df_stable[TARGET_COL] == 3.0) &
        (df_stable['BMI']      >= 23.0)
    ].copy()

    preds     = model.predict(class3_pool[X_FEATURES])
    confirmed = class3_pool[preds == 3]

    if len(confirmed) == 0:
        print("⚠️  모델 확인 등급 3 환자 없음. 전체 등급 3으로 대체.")
        confirmed = class3_pool

    sample = confirmed.sample(1, random_state=42)
    query  = sample[X_FEATURES]
    print(f"추출 환자 — 실제 등급: {int(sample[TARGET_COL].values[0])}, "
          f"예측 등급: {int(model.predict(query)[0])}, "
          f"BMI: {query['BMI'].values[0]:.2f}")

    # ── 2. 범주형 피처 메타데이터 구성 ──────────────────────────────────────
    feature_metadata = {
        col: sorted(df_stable[col].unique().tolist())
        for col in CAT_FEATURES if col in df_stable.columns
    }

    # ── 3. LLM 가드레일 생성 ────────────────────────────────────────────────
    print("GPT-4o-mini 임상 가드레일 에이전트 호출 중...")
    agent_plan = get_clinical_guardrails(query, group_name, X_FEATURES, feature_metadata)
    print("추론 요약:", agent_plan.get('reasoning', '')[:300], "...")

    # ── 4. 물리적 보정 레이어 (하드 규칙 — LLM 출력 무조건 덮어씀) ──────────
    cur_bmi    = float(query.iloc[0]['BMI'])
    cur_waist  = float(query.iloc[0]['WaistCirc'])
    cur_weight = float(query.iloc[0]['Weight'])
    cur_energy = float(query.iloc[0]['Energy_kcal'])
    cur_sodium = float(query.iloc[0]['Sodium_mg'])
    cur_carb   = float(query.iloc[0]['Carb_g'])
    cur_sugar  = float(query.iloc[0]['Sugar_g'])
    cur_prot   = float(query.iloc[0]['Protein_g'])
    cur_pot    = float(query.iloc[0]['Potassium_mg'])
    cur_fiber  = float(query.iloc[0]['Fiber_g'])

    # Step 1: LLM 범위를 기초로, 모든 하한 0 이상 보정
    valid_guardrails = {}
    for feat in X_FEATURES:
        cur = float(query.iloc[0][feat])
        if feat in agent_plan['guardrail_ranges']:
            lo = max(float(agent_plan['guardrail_ranges'][feat][0]), 0.0)
            hi = max(float(agent_plan['guardrail_ranges'][feat][1]), lo)
        else:
            lo, hi = max(cur * 0.7, 0.0), cur * 1.3
        valid_guardrails[feat] = [lo, hi]

    # 규칙 A: 신체계측 지표 일관성 (BMI·허리둘레·체중은 동일 방향 변화)
    # 범위: [현재값×0.85, 현재값×1.00] — 감소만 허용
    for feat, cur in [('BMI', cur_bmi), ('WaistCirc', cur_waist), ('Weight', cur_weight)]:
        valid_guardrails[feat] = [round(cur * 0.85, 4), round(cur * 1.00, 4)]

    weight_floor_ratio = valid_guardrails['Weight'][0] / cur_weight
    valid_guardrails['WaistCirc'][0] = max(
        valid_guardrails['WaistCirc'][0],
        round(cur_waist * weight_floor_ratio, 4)
    )
    valid_guardrails['BMI'][0] = max(
        valid_guardrails['BMI'][0],
        round(cur_bmi * weight_floor_ratio, 4)
    )

    # 규칙 B: 에너지 하한 70%, 상한 현재값 (열량 증가 불허)
    valid_guardrails['Energy_kcal'] = [
        round(max(cur_energy * 0.70, 500.0), 4),
        round(cur_energy, 4)
    ]

    # 규칙 C: 나트륨 하한 — 저나트륨혈증 방지 (WHO 기준 800mg/일)
    sodium_floor   = round(max(cur_sodium * 0.60, 800.0), 4)
    sodium_ceiling = round(cur_sodium * 0.90, 4)
    valid_guardrails['Sodium_mg'] = [sodium_floor, sodium_ceiling]

    # 규칙 D: 나트륨 감소 시 탄수화물·당류 상한 = 현재값 (보상적 섭취 차단)
    valid_guardrails['Carb_g'][1]  = round(cur_carb,  4)
    valid_guardrails['Sugar_g'][1] = round(cur_sugar, 4)

    # 규칙 E: 영양소 결핍 방지 최솟값
    # 단백질: 현재의 60% 또는 30g (근감소 예방)
    # 칼륨:   현재의 50% 또는 500mg (심장 안전)
    # 탄수화물: 현재의 20% 또는 30g (뇌 포도당 최소치)
    # 식이섬유: 현재의 30% 또는 5g (장 건강 최소치)
    valid_guardrails['Protein_g'][0]    = round(max(cur_prot  * 0.60, 30.0), 4)
    valid_guardrails['Potassium_mg'][0] = round(max(cur_pot   * 0.50, 500.0), 4)
    valid_guardrails['Carb_g'][0]       = round(max(cur_carb  * 0.20, 30.0), 4)
    valid_guardrails['Fiber_g'][0]      = round(max(cur_fiber * 0.30, 5.0),  4)

    # 최종 점검: lo > hi 방지
    for feat in X_FEATURES:
        lo, hi = valid_guardrails[feat]
        if lo > hi:
            cur = float(query.iloc[0][feat])
            valid_guardrails[feat] = [round(cur * 0.85, 4), round(cur * 1.0, 4)]

    print("\n최종 가드레일 범위 (앞 6개 미리보기):")
    for k, v in list(valid_guardrails.items())[:6]:
        print(f"  {k}: {v}")

    # ── 5. DiCE 설정 ────────────────────────────────────────────────────────
    # 주의: DiCE genetic 방식은 범주형 선언 시 탐색 공간 축소 문제 발생
    # → 모든 피처를 연속형으로 선언하고, permitted_range로 정수 범위 제한
    d   = dice_ml.Data(
            dataframe=df_stable[X_FEATURES + [TARGET_COL]],
            continuous_features=X_FEATURES,
            outcome_name=TARGET_COL,
          )
    m   = dice_ml.Model(model=model, backend="sklearn")
    exp = dice_ml.Dice(d, m, method="genetic")

    # ── 6. 단계적 폴백: 등급 0 → 1 → 2 ────────────────────────────────────
    # 목표 등급 0(완전 정상화) 불가 시 등급 1(고혈압 관리) → 등급 2(당뇨 관리) 순차 시도
    # 가드레일 제약은 모든 단계에서 동일하게 유지 (안전성 불변)
    fallback_targets = {0: "정상", 1: "고혈압만", 2: "당뇨만"}
    for target_class, target_name in fallback_targets.items():
        print(f"\n▶ [{target_name}] (등급 {target_class}) 반사실 탐색 중...")
        for attempt in range(15):
            try:
                cf = exp.generate_counterfactuals(
                    query,
                    total_CFs=4,
                    desired_class=target_class,
                    features_to_vary=VARY_FEATURES,
                    permitted_range=valid_guardrails,
                    proximity_weight=0.2,
                    sparsity_weight=0.1,
                )
                if cf and len(cf.cf_examples_list[0].final_cfs_df) > 0:
                    cf_df = cf.cf_examples_list[0].final_cfs_df.copy()

                    # 사후처리 1: 범주형 변수 → 가장 가까운 유효 정수로 반올림
                    vary_cat = [f for f in CAT_FEATURES if f not in FIXED_FEATURES]
                    for col in vary_cat:
                        if col in cf_df.columns:
                            valid_vals = sorted(df_stable[col].dropna().unique().tolist())
                            cf_df[col] = cf_df[col].apply(
                                lambda v: min(valid_vals, key=lambda x: abs(x - v))
                            )

                    # 사후처리 2: 행별 신체계측 지표 결합 강제 적용
                    # 가장 크게 변한 지표의 감소 비율을 기준으로
                    # 나머지 지표를 동일 비율로 조정 → 경로별 다양성 유지
                    orig_bmi   = float(query.iloc[0]['BMI'])
                    orig_waist = float(query.iloc[0]['WaistCirc'])
                    orig_wt    = float(query.iloc[0]['Weight'])

                    def enforce_anthro_coupling(row):
                        b, w, wt = row['BMI'], row['WaistCirc'], row['Weight']
                        changed = (
                            abs(b  - orig_bmi)   > 0.01 or
                            abs(w  - orig_waist) > 0.01 or
                            abs(wt - orig_wt)    > 0.01
                        )
                        if not changed:
                            return row
                        r_bmi   = b  / orig_bmi
                        r_waist = w  / orig_waist
                        r_wt    = wt / orig_wt
                        r_target = max(0.85, min(min(r_bmi, r_waist, r_wt), 1.0))
                        if r_bmi   > r_target: row['BMI']       = round(orig_bmi   * r_target, 4)
                        if r_waist > r_target: row['WaistCirc'] = round(orig_waist * r_target, 4)
                        if r_wt    > r_target: row['Weight']    = round(orig_wt    * r_target, 4)
                        return row

                    cf_df = cf_df.apply(enforce_anthro_coupling, axis=1)
                    cf.cf_examples_list[0].final_cfs_df = cf_df

                    print(f"✅ [{target_name}] 경로 발견 (시도 {attempt + 1}회차)!")
                    cf.visualize_as_dataframe(show_only_changes=True)
                    return cf, target_class   # 달성 등급도 함께 반환
            except Exception:
                continue
        print(f"⚠️  [{target_name}] 도달 불가. 다음 목표로 폴백.")

    print("✖ 모든 목표 소진. 유효한 반사실 없음.")
    return None, None

# %%
# ## 6. 전체 파이프라인 실행 (4개 그룹)

results = {}   # 그룹명 → (cf 결과, 달성 등급)

for group_name, cfg in AGEGROUP_CONFIG.items():
    mdl = models.get(group_name)
    if mdl is None:
        print(f"\n[{group_name}] 모델 없음. 건너뜀.")
        results[group_name] = (None, None)
        continue

    age_grp = 0.0 if cfg['age_min'] < 60 else 1.0
    cf, achieved_class = run_dice_pipeline(
        model       = mdl,
        df_input    = df_final,
        age_group   = age_grp,
        gender_code = cfg['sex_code'],
        group_name  = group_name,
    )
    results[group_name] = (cf, achieved_class)

# %%
# ## 7. 가드레일 적용 전 비교 기준선 — 비제약 DiCE (논문 Table 6용)

def run_dice_no_guardrail(model, df_input: pd.DataFrame,
                           age_group: float, gender_code: float, group_name: str):
    """
    가드레일 없이 순수 DiCE를 동일 환자에게 적용.
    수치적 환각(numerical hallucination) 및 개입 충돌 사례 도출 → Table 6 비교 데이터.
    """
    print(f"\n{'='*20} [{group_name}] 비제약 DiCE 기준선 {'='*20}")

    df_stable = df_input.copy().astype(float)
    class3_pool = df_stable[
        (df_stable['AgeGroup'] == age_group) &
        (df_stable['Sex']      == gender_code) &
        (df_stable[TARGET_COL] == 3.0) &
        (df_stable['BMI']      >= 23.0)
    ].copy()
    preds     = model.predict(class3_pool[X_FEATURES])
    confirmed = class3_pool[preds == 3]
    if len(confirmed) == 0:
        confirmed = class3_pool
    sample = confirmed.sample(1, random_state=42)
    query  = sample[X_FEATURES]

    d   = dice_ml.Data(
            dataframe=df_stable[X_FEATURES + [TARGET_COL]],
            continuous_features=X_FEATURES,
            outcome_name=TARGET_COL,
          )
    m   = dice_ml.Model(model=model, backend="sklearn")
    exp = dice_ml.Dice(d, m, method="genetic")

    try:
        cf = exp.generate_counterfactuals(
            query,
            total_CFs=4,
            desired_class=0,
            features_to_vary=VARY_FEATURES,
            proximity_weight=0.2,
            sparsity_weight=0.1,
        )
        if cf and len(cf.cf_examples_list[0].final_cfs_df) > 0:
            print("비제약 DiCE 결과 (수치 환각 여부 확인):")
            cf.visualize_as_dataframe(show_only_changes=True)
            return cf
    except Exception as e:
        print(f"비제약 탐색 실패: {e}")
    return None

# 4개 그룹 기준선 실행
baselines = {}
for group_name, cfg in AGEGROUP_CONFIG.items():
    mdl = models.get(group_name)
    if mdl is None:
        baselines[group_name] = None
        continue
    age_grp = 0.0 if cfg['age_min'] < 60 else 1.0
    baselines[group_name] = run_dice_no_guardrail(
        model       = mdl,
        df_input    = df_final,
        age_group   = age_grp,
        gender_code = cfg['sex_code'],
        group_name  = group_name,
    )

# %%
# ## 8. 결과 JSON 저장

def save_results_to_json(results: dict, baselines: dict,
                          filename: str = "실험결과_연령성별층화.json"):
    """
    4개 그룹 반사실 결과 및 비제약 기준선을 JSON으로 저장.
    'achieved_class' 필드로 폴백 깊이 기록 → 논문 Table 4·5·6 원자료.
    """
    export = {
        "메타데이터": {
            "설명":         "동반이환 만성질환 계약자 개인화 건강 개입 경로 생성 모형",
            "모델":         "XGBoost + GPT-4o-mini + DiCE",
            "데이터":       "KNHANES 2020–2024",
            "층화기준":     "연령군(중장년/고령) × 성별(남/여) 4개 모델",
            "경로수_per환자": 4,
        },
        "결과": [],
        "비제약_기준선": [],   # Table 6 비교 데이터
    }

    for group_name, (cf, achieved_class) in results.items():
        if cf is not None:
            entry = {
                "그룹":         group_name,
                "상태":         "성공",
                "달성등급":     int(achieved_class),
                "달성등급설명": {0: "정상", 1: "고혈압만", 2: "당뇨만"}.get(achieved_class, "불명"),
                "원본환자":     cf.cf_examples_list[0].test_instance_df.to_dict(orient='records')[0],
                "반사실경로":   cf.cf_examples_list[0].final_cfs_df.to_dict(orient='records'),
            }
        else:
            entry = {
                "그룹":   group_name,
                "상태":   "실패",
                "메시지": "가드레일 제약 내에서 유효한 반사실 미발견.",
            }
        export["결과"].append(entry)

    for group_name, cf in baselines.items():
        if cf is not None:
            export["비제약_기준선"].append({
                "그룹":       group_name,
                "원본환자":   cf.cf_examples_list[0].test_instance_df.to_dict(orient='records')[0],
                "반사실경로": cf.cf_examples_list[0].final_cfs_df.to_dict(orient='records'),
            })

    with open(filename, 'w', encoding='utf-8') as f:
        json.dump(export, f, ensure_ascii=False, indent=4)
    print(f"✅ 결과 저장 완료: {filename}")

save_results_to_json(results, baselines)


HE_DM_HbA1c 분포:
 HE_DM_HbA1c
-999.0    17438
 0.0      12810
 1.0       4392
Name: count, dtype: int64

HE_HP 분포:
 HE_HP
-999.0    16734
 0.0      11964
 1.0       5942
Name: count, dtype: int64

위험등급 분포 (전체):
integrated_target
0    6381
1    1370
2     649
3    1338
Name: count, dtype: int64

연령군 분포 (0=중장년 40–59세, 1=고령 60세+):
age_group
0    3360
1    2979
Name: count, dtype: int64
>>> 전처리 완료. CSV 저장됨.

df_final 형태: (6339, 34)
문자형 컬럼 존재 여부: False

[연령군 × 성별 × 위험등급 분포]
RiskClass      0.0  1.0  2.0  3.0
AgeGroup Sex                     
0.0      1.0   615  196  106  171
         2.0  1785  242  143  102
1.0      1.0   260  344  194  526
         2.0   465  505  170  515

==================== [중장년_남성] 모델 학습 ====================
  표본 수: 1088명
  위험등급 분포:
RiskClass
0.0    615
1.0    196
2.0    106
3.0    171
Name: count, dtype: int64

[중장년_남성] 분류 성능 보고서:
              precision    recall  f1-score   support

           0       0.72      0.84      0.77       123
           1       0.46      

100%|████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  3.60it/s]

✅ [정상] 경로 발견 (시도 1회차)!
Query instance (original outcome : 3)


,BMI,WaistCirc,Weight,Energy_kcal,Carb_g,Sugar_g,Sodium_mg,Fat_g,SaturatedFat_g,Fiber_g,Potassium_mg,Protein_g,ObesityStatus,WeightChangeStatus,WeightLossAmount,WeightGainAmount,DrinkingFrequency,DrinkingAmount,SmokingStatus,VigorousActivity_Work,VigorousActivity_Leisure,ModerateActivity_Work,WalkingActivity,AerobicActivityRate,BreakfastFrequency,StressLevel,StressAwarenessRate,PersonalIncomeQuartile,HouseholdIncomeQuartile,EducationLevel,HealthScreeningStatus,RiskClass
0,28.102524,97.800003,84.400002,4561.587891,219.612122,28.216547,6447.066895,163.32608,55.126991,32.326542,5388.150391,241.388077,4.0,1.0,8.0,8.0,5.0,5.0,1.0,2.0,2.0,1.0,2.0,1.0,2.0,3.0,0.0,2.0,3.0,3.0,1.0,3



Diverse Counterfactual set (new outcome: 0)


...

비제약 DiCE 결과 (수치 환각 여부 확인):
Query instance (original outcome : 3)


,BMI,WaistCirc,Weight,Energy_kcal,Carb_g,Sugar_g,Sodium_mg,Fat_g,SaturatedFat_g,Fiber_g,Potassium_mg,Protein_g,ObesityStatus,WeightChangeStatus,WeightLossAmount,WeightGainAmount,DrinkingFrequency,DrinkingAmount,SmokingStatus,VigorousActivity_Work,VigorousActivity_Leisure,ModerateActivity_Work,WalkingActivity,AerobicActivityRate,BreakfastFrequency,StressLevel,StressAwarenessRate,PersonalIncomeQuartile,HouseholdIncomeQuartile,EducationLevel,HealthScreeningStatus,RiskClass
0,28.080086,103.400002,70.900002,1110.694702,209.233612,23.996782,2448.812256,10.122841,3.430352,17.67223,1841.021118,43.439129,4.0,1.0,8.0,8.0,1.0,8.0,8.0,2.0,2.0,1.0,2.0,0.0,1.0,2.0,1.0,1.0,1.0,1.0,1.0,3



Diverse Counterfactual set (new outcome: 0)


,BMI,WaistCirc,Weight,Energy_kcal,Carb_g,Sugar_g,Sodium_mg,Fat_g,SaturatedFat_g,Fiber_g,Potassium_mg,Protein_g,ObesityStatus,WeightChangeStatus,WeightLossAmount,WeightGainAmount,DrinkingFrequency,DrinkingAmount,SmokingStatus,VigorousActivity_Work,VigorousActivity_Leisure,ModerateActivity_Work,WalkingActivity,AerobicActivityRate,BreakfastFrequency,StressLevel,StressAwarenessRate,PersonalIncomeQuartile,HouseholdIncomeQuartile,EducationLevel,HealthScreeningStatus,RiskClass
0,21.231005,54.3,52.8,160.404,197.841385,0.35505927,-,46.228371,12.64693165,0.4251,-,55.1748619,2.0,-,-,-,-,-,-,-,-,2.0,-,-,-,-,-,-,-,-,-,0.0
0,19.878525,67.8,60.6,1319.9012,198.394348,45.85150146,-,34.105797,13.50623894,17.45243,-,56.1345711,2.0,-,-,-,2.0,1.0,-,-,-,-,-,-,-,-,-,-,-,-,-,0.0
0,20.959476,54.3,60.6,-,198.394348,46.42942429,46.725,34.105797,13.50623894,17.45243,-,1.7118599,2.0,-,-,-,2.0,1.0,-,-,-,-,-,-,-,-,-,-,-,-,-,0.0
0,13.544199,54.3,52.8,-,232.994507,34.39852142,-,46.228371,4.04298353,22.97421,-,1.7118599,2.0,-,1.0,-,3.0,-,-,-,-,-,-,-,-,-,-,-,-,-,-,0.0


✅ 결과 저장 완료: 실험결과_연령성별층화.json
